# GOV-01 V2: masked multi-label MobileNetV2 baseline

This notebook is the first V2 multi-label training experiment. It predicts whether each of these seven road conditions is visible in an image: `crack`, `pothole`, `repaired_road`, `manhole_cover`, `unpaved_road`, `road_marking`, and `speed_bump`.

Important rule: some sources only confirm one condition. For those images, the other conditions are **unknown**, not **no**. The masked loss below learns only from labels that the source actually knows.

This notebook uses **train** and **validation** only. The protected test split must not be loaded, inspected, or used to make training decisions.

## Before you run

1. Upload the ready `v2_multilabel.zip` archive to `My Drive/GOV-01/`. Do not add the ZIP to GitHub.
2. The next cell is already set for this Google Drive archive. Do not change its paths or data mode.
3. In Colab, select **Runtime → Change runtime type → T4 GPU** before running the notebook.
4. Run the cells from top to bottom.

Training metrics, plots, and checkpoints stay in temporary Colab storage under `/content/v2_multilabel_output`. Download the final output ZIP before the runtime resets.

In [ ]:
# Dataset access. Change only these settings when needed.
from pathlib import Path
import csv
import shutil
import zipfile

DATA_MODE = 'drive_archive'  # ready ZIP in Google Drive; other modes are only alternatives
DRIVE_DATA_DIRECTORY = Path('/content/drive/MyDrive/GOV-01/v2_multilabel')
DRIVE_ARCHIVE = Path('/content/drive/MyDrive/GOV-01/v2_multilabel.zip')
LOCAL_EXTRACT_DIRECTORY = Path('/content/v2_multilabel')
OUTPUT_DIRECTORY = Path('/content/v2_multilabel_output')

if DATA_MODE in {'drive_directory', 'drive_archive'}:
    from google.colab import drive
    drive.mount('/content/drive')

def split_is_complete(dataset_root, split_name):
    split_root = dataset_root / split_name
    labels_path = split_root / 'labels.csv'
    images_root = split_root / 'images'
    if not labels_path.is_file() or not images_root.is_dir():
        return False
    with labels_path.open(newline='', encoding='utf-8') as source:
        for row in csv.DictReader(source):
            relative_image = Path(row['materialized_image'].replace(chr(92), '/'))
            if not (split_root / relative_image).is_file():
                return False
    return True

def training_data_is_complete(dataset_root):
    return all(split_is_complete(dataset_root, split_name) for split_name in ('train', 'validation'))

def extract_clean_copy(archive_path):
    if training_data_is_complete(LOCAL_EXTRACT_DIRECTORY):
        print('Using verified existing Colab extraction:', LOCAL_EXTRACT_DIRECTORY)
        return LOCAL_EXTRACT_DIRECTORY
    if LOCAL_EXTRACT_DIRECTORY.exists():
        print('Removing incomplete temporary Colab extraction:', LOCAL_EXTRACT_DIRECTORY)
        shutil.rmtree(LOCAL_EXTRACT_DIRECTORY)
    print('Extracting the dataset ZIP to temporary Colab storage. This can take several minutes.')
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(LOCAL_EXTRACT_DIRECTORY.parent)
    if not training_data_is_complete(LOCAL_EXTRACT_DIRECTORY):
        raise RuntimeError('Extraction finished but the training data is incomplete. Rerun this cell once; it will clean the temporary copy first.')
    print('Verified complete train and validation extraction:', LOCAL_EXTRACT_DIRECTORY)
    return LOCAL_EXTRACT_DIRECTORY

if DATA_MODE == 'drive_directory':
    DATA_ROOT = DRIVE_DATA_DIRECTORY
    if not training_data_is_complete(DATA_ROOT):
        raise FileNotFoundError(f'Google Drive training directory is incomplete: {DATA_ROOT}')
elif DATA_MODE == 'drive_archive':
    if not DRIVE_ARCHIVE.is_file():
        raise FileNotFoundError(f'Google Drive archive was not found: {DRIVE_ARCHIVE}')
    DATA_ROOT = extract_clean_copy(DRIVE_ARCHIVE)
elif DATA_MODE == 'uploaded_archive':
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload exactly one v2_multilabel ZIP archive.')
    archive_path = Path(next(iter(uploaded)))
    DATA_ROOT = extract_clean_copy(archive_path)
else:
    raise ValueError('DATA_MODE must be drive_directory, drive_archive, or uploaded_archive.')
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
print('Training data root:', DATA_ROOT)
print('Colab output folder:', OUTPUT_DIRECTORY)
print('Protected test is deliberately not loaded by this notebook.')

In [ ]:
# Imports, fixed experiment settings, and GPU check.
import csv
import json
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import f1_score

SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 20
CONDITIONS = ('crack', 'pothole', 'repaired_road', 'manhole_cover', 'unpaved_road', 'road_marking', 'speed_bump')

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
print('TensorFlow:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))
if not tf.config.list_physical_devices('GPU'):
    raise RuntimeError('No GPU was found. In Colab choose Runtime → Change runtime type → T4 GPU, reconnect, and run this notebook from the top.')

In [ ]:
# Read labels. A mask value of 1 means the source knows that label; 0 means unknown.
def read_split(split_name):
    split_root = DATA_ROOT / split_name
    paths, labels, masks = [], [], []
    with (split_root / 'labels.csv').open(newline='', encoding='utf-8') as source:
        for row in csv.DictReader(source):
            relative_image = Path(row['materialized_image'].replace(chr(92), '/'))
            image_path = split_root / relative_image
            if not image_path.is_file():
                raise FileNotFoundError(f'Missing materialized image: {image_path}')
            label_row, mask_row = [], []
            for condition in CONDITIONS:
                known = int(row[f'{condition}_known'])
                value = row[f'{condition}_present']
                if known == 0 and value != '':
                    raise ValueError(f"{row['record_id']} gives a value for unknown {condition}")
                label_row.append(float(value) if known else 0.0)
                mask_row.append(float(known))
            paths.append(str(image_path))
            labels.append(label_row)
            masks.append(mask_row)
    return np.array(paths), np.asarray(labels, dtype='float32'), np.asarray(masks, dtype='float32')

train_paths, train_labels, train_masks = read_split('train')
validation_paths, validation_labels, validation_masks = read_split('validation')

def print_coverage(name, labels, masks):
    print(f'\n{name}: {len(labels)} images')
    for index, condition in enumerate(CONDITIONS):
        known = int(masks[:, index].sum())
        positive = int((labels[:, index] * masks[:, index]).sum())
        print(f'{condition:16} known={known:5}  positive={positive:5}')

print_coverage('Train', train_labels, train_masks)
print_coverage('Validation', validation_labels, validation_masks)

In [ ]:
# Build efficient input pipelines. The protected test split is never passed here.
AUTOTUNE = tf.data.AUTOTUNE

def decode_image(path):
    image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    return tf.image.resize(tf.cast(image, tf.float32), IMAGE_SIZE)

def make_dataset(paths, labels, masks, training):
    packed_targets = np.concatenate([labels, masks], axis=1)
    dataset = tf.data.Dataset.from_tensor_slices((paths, packed_targets))
    if training:
        dataset = dataset.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.map(lambda path, target: (decode_image(path), target), num_parallel_calls=AUTOTUNE)
    return dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_dataset = make_dataset(train_paths, train_labels, train_masks, training=True)
validation_dataset = make_dataset(validation_paths, validation_labels, validation_masks, training=False)
print('Train batches:', tf.data.experimental.cardinality(train_dataset).numpy())
print('Validation batches:', tf.data.experimental.cardinality(validation_dataset).numpy())

In [ ]:
# The custom loss ignores unknown labels instead of treating them as negative examples.
@tf.keras.utils.register_keras_serializable(package='gov01')
def masked_binary_crossentropy(packed_targets, probabilities):
    label_count = len(CONDITIONS)
    labels = packed_targets[:, :label_count]
    masks = packed_targets[:, label_count:]
    per_label_loss = tf.keras.backend.binary_crossentropy(labels, probabilities)
    return tf.math.divide_no_nan(tf.reduce_sum(per_label_loss * masks, axis=1), tf.reduce_sum(masks, axis=1))

@tf.keras.utils.register_keras_serializable(package='gov01')
class MaskedBinaryAccuracy(tf.keras.metrics.Metric):
    def __init__(self, name='masked_binary_accuracy', **kwargs):
        super().__init__(name=name, **kwargs)
        self.correct = self.add_weight(name='correct', initializer='zeros')
        self.count = self.add_weight(name='count', initializer='zeros')

    def update_state(self, packed_targets, probabilities, sample_weight=None):
        label_count = len(CONDITIONS)
        labels = packed_targets[:, :label_count]
        masks = packed_targets[:, label_count:]
        predictions = tf.cast(probabilities >= 0.5, tf.float32)
        self.correct.assign_add(tf.reduce_sum(tf.cast(tf.equal(labels, predictions), tf.float32) * masks))
        self.count.assign_add(tf.reduce_sum(masks))

    def result(self):
        return tf.math.divide_no_nan(self.correct, self.count)

    def reset_state(self):
        self.correct.assign(0.0)
        self.count.assign(0.0)

class ValidationMaskedMacroF1(tf.keras.callbacks.Callback):
    def __init__(self, images, labels, masks):
        super().__init__()
        self.images = images
        self.labels = labels
        self.masks = masks
        self.history = []

    def on_epoch_end(self, epoch, logs=None):
        probabilities = self.model.predict(self.images, verbose=0)
        scores = []
        for index in range(len(CONDITIONS)):
            known = self.masks[:, index].astype(bool)
            scores.append(f1_score(self.labels[known, index], probabilities[known, index] >= 0.5, zero_division=0))
        macro_f1 = float(np.mean(scores))
        self.history.append(macro_f1)
        print(f' — val_masked_macro_f1: {macro_f1:.4f}')

validation_macro_f1 = ValidationMaskedMacroF1(validation_dataset.map(lambda image, target: image), validation_labels, validation_masks)

In [ ]:
# Frozen ImageNet-pretrained MobileNetV2: the first controlled baseline.
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal', seed=SEED),
    tf.keras.layers.RandomRotation(0.05, seed=SEED),
    tf.keras.layers.RandomContrast(0.10, seed=SEED),
], name='training_only_augmentation')

base = tf.keras.applications.MobileNetV2(
    include_top=False, weights='imagenet', input_shape=(*IMAGE_SIZE, 3)
)
base.trainable = False

inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3), name='road_image')
x = augmentation(inputs)  # Keras applies this only while training.
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.30, seed=SEED)(x)
outputs = tf.keras.layers.Dense(len(CONDITIONS), activation='sigmoid', name='condition_probabilities')(x)
model = tf.keras.Model(inputs, outputs, name='v2_multilabel_frozen_mobilenetv2')
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss=masked_binary_crossentropy, metrics=[MaskedBinaryAccuracy()])
model.summary()

In [ ]:
# Train with validation only. The protected test is still untouched.
checkpoint_path = OUTPUT_DIRECTORY / 'v2_multilabel_frozen_mobilenetv2_best.keras'
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(checkpoint_path, monitor='val_loss', save_best_only=True),
    validation_macro_f1,
]

started = time.time()
history = model.fit(train_dataset, validation_data=validation_dataset, epochs=EPOCHS, callbacks=callbacks, verbose=1)
print(f'Training time: {time.time() - started:.1f} seconds')
print('Best validation checkpoint:', checkpoint_path)

In [ ]:
# Save validation-only evidence: metrics and learning curves.
best_model = tf.keras.models.load_model(
    checkpoint_path,
    custom_objects={
        'masked_binary_crossentropy': masked_binary_crossentropy,
        'MaskedBinaryAccuracy': MaskedBinaryAccuracy,
    },
)
validation_probabilities = best_model.predict(validation_dataset.map(lambda image, target: image), verbose=1)
per_condition_f1 = {}
for index, condition in enumerate(CONDITIONS):
    known = validation_masks[:, index].astype(bool)
    per_condition_f1[condition] = float(f1_score(validation_labels[known, index], validation_probabilities[known, index] >= 0.5, zero_division=0))

validation_metrics = {
    'experiment': 'v2_multilabel_frozen_mobilenetv2',
    'selection_data': 'validation only',
    'protected_test_loaded': False,
    'conditions': list(CONDITIONS),
    'threshold': 0.5,
    'masked_validation_macro_f1': float(np.mean(list(per_condition_f1.values()))),
    'per_condition_f1': per_condition_f1,
    'epochs_completed': len(history.history['loss']),
}
(OUTPUT_DIRECTORY / 'v2_multilabel_validation_metrics.json').write_text(json.dumps(validation_metrics, indent=2) + '\n')
best_model.save(OUTPUT_DIRECTORY / 'v2_multilabel_frozen_mobilenetv2_final.keras')
print(json.dumps(validation_metrics, indent=2))

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='validation loss')
plt.xlabel('Epoch')
plt.ylabel('Masked binary cross-entropy')
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(validation_macro_f1.history, label='validation masked macro F1')
plt.xlabel('Epoch')
plt.ylabel('F1')
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIRECTORY / 'v2_multilabel_learning_curve.png', dpi=150)
plt.show()

In [ ]:
# Run only after reviewing the validation result. This downloads your local Colab output.
from google.colab import files
archive_path = shutil.make_archive('/content/v2_multilabel_frozen_mobilenetv2_output', 'zip', OUTPUT_DIRECTORY)
print('Downloading:', archive_path)
files.download(archive_path)
# Do not evaluate the protected test until you deliberately close model selection.